## tl;dr

在原有 baseline 恢复反事实（300.358 ms）中，1131 预测不碰撞，0 m 余量为 +5.495 m。把实际日志中的异常超额时延移入首次响应链后：Fusion 单独加入、Planning 单独加入、二者按实际重叠加入，模型均预测碰撞；对应预测撞击速度约为 7.35、7.71 和 7.87 m/s。该结果是模型敏感性分析，不表示这些晚期异常在真实时间线上造成了首次制动延迟。

## Context & Methods

### Key Assumptions

- 保留现有物理模型：制动能力取1131实际 t2 到碰撞的能量等效减速度。
- D_delay 始终使用 Localization 速度对墙钟时间的梯形积分。
- 当前 baseline 反事实已包含正常模块处理，因此只新增异常超额：Fusion 输出间隔减相邻源周期，Planning 最大周期减窗口中位数。
- 实际异常高度重叠，主联合情景使用两个异常超额区间的时间并集；串行求和仅作为保守上界。
- 若候选 t2 晚于实际 t2，观测轨迹已受制动影响，改用实际 v2 恒速延长作为无制动保守假设。

## Data

数据来自 `run_level_metrics.csv`、`counterfactual_model.csv`、1131 Localization/Perception/Planning/CollisionSensor 原始日志。执行时工作目录应为项目根目录。

In [1]:
import csv, math, sys
from pathlib import Path
from zoneinfo import ZoneInfo
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'realtime_defect_report' / 'scripts'))
import analyze_realtime_defects as analysis

config = analysis.make_config()
timezone = ZoneInfo(config['analysis']['timezone'])
spec = next(s for s in analysis.core.discover_runs(config) if s.run_id == '202607271131')
parsed = analysis.core.parse_run(spec, config, timezone)
with (ROOT / 'realtime_defect_report/tables/run_level_metrics.csv').open(encoding='utf-8-sig', newline='') as handle:
    observed = next(row for row in csv.DictReader(handle) if row['run_id'] == '202607271131')

value = lambda key: float(observed[key])
t1 = value('t1_wall_s')
t2_actual = value('t2_wall_s')
actual_latency_ms = value('T_e2e_data_observed_ms')
d1_m = value('D1_clear_data_observed_m')
actual_d_delay_m = value('D_delay_wall_integral_data_observed_m')
actual_d2_m = value('D2_clear_data_observed_m')
v2_actual_mps = value('v2_data_observed_mps')
impact_actual_mps = value('impact_speed_data_observed_mps')
collision_s = float(parsed.collision['time_s'])
actual_path_to_contact_m = analysis.core.integrate_speed(parsed.localization, t2_actual, collision_s)
equivalent_deceleration_mps2 = (v2_actual_mps**2 - impact_actual_mps**2) / (2 * actual_path_to_contact_m)
contact_correction_m = actual_path_to_contact_m - actual_d2_m
baseline_ms = 300.3580570220947
print(f'baseline_ms={baseline_ms:.6f}, actual_latency_ms={actual_latency_ms:.6f}')
print(f'D1={d1_m:.6f} m, actual D_delay={actual_d_delay_m:.6f} m, actual D2={actual_d2_m:.6f} m')
print(f'equivalent_deceleration={equivalent_deceleration_mps2:.6f} m/s^2, contact_correction={contact_correction_m:.6f} m')

baseline_ms=300.358057, actual_latency_ms=799.635887
D1=38.258206 m, actual D_delay=13.432028 m, actual D2=24.826178 m
equivalent_deceleration=4.466860 m/s^2, contact_correction=2.322178 m


In [2]:
fusion_gap_ms = 507.43889808654785
fusion_source_period_ms = 99.78365898132324
fusion_excess_ms = fusion_gap_ms - fusion_source_period_ms
planning_total_ms = 472.595
planning_window_median_ms = 14.0986
planning_excess_ms = planning_total_ms - planning_window_median_ms

planning_start_s = 1785123123.949573
planning_excess_start_s = planning_start_s + planning_window_median_ms / 1000
planning_total_end_s = planning_start_s + planning_total_ms / 1000
fusion_excess_start_s = 1785123124.0374167
fusion_gap_end_s = 1785123124.445072
both_union_ms = (max(planning_total_end_s, fusion_gap_end_s) - min(planning_excess_start_s, fusion_excess_start_s)) * 1000
both_serial_ms = fusion_excess_ms + planning_excess_ms

anomaly_inputs = pd.DataFrame([
    {'component': 'Fusion', 'observed_abnormal_ms': fusion_gap_ms, 'normal_reference_ms': fusion_source_period_ms, 'excess_ms': fusion_excess_ms},
    {'component': 'Planning', 'observed_abnormal_ms': planning_total_ms, 'normal_reference_ms': planning_window_median_ms, 'excess_ms': planning_excess_ms},
    {'component': 'Both: actual-time union', 'observed_abnormal_ms': math.nan, 'normal_reference_ms': math.nan, 'excess_ms': both_union_ms},
    {'component': 'Both: serial upper bound', 'observed_abnormal_ms': math.nan, 'normal_reference_ms': math.nan, 'excess_ms': both_serial_ms},
])
print(anomaly_inputs.to_string(index=False))

           component  observed_abnormal_ms  normal_reference_ms  excess_ms
              Fusion            507.438898            99.783659 407.655239
             Planning            472.595000            14.098600 458.496400
Both: actual-time union                   NaN                  NaN 481.400251
  Both: serial upper bound                   NaN                  NaN 866.151639


## Results

以下结果全部是 `model/predicted`。候选 t2 不晚于实际 t2 时，速度和 D_delay 直接取1131的观测墙钟轨迹；只有串行上界越过实际 t2，使用恒定 v2 的无制动延长。

In [3]:
def evaluate_scenario(name, added_delay_ms):
    response_latency_ms = baseline_ms + added_delay_ms
    candidate_t2_s = t1 + response_latency_ms / 1000
    if candidate_t2_s <= t2_actual:
        state = analysis.core.interpolate_sample(parsed.localization, candidate_t2_s)
        brake_start_speed_mps = float(state['speed_mps'])
        d_delay_model_m = analysis.core.integrate_speed(parsed.localization, t1, candidate_t2_s)
        extension_method = 'observed pre-t2 wall-clock speed trace'
    else:
        extra_after_actual_t2_s = candidate_t2_s - t2_actual
        brake_start_speed_mps = v2_actual_mps
        d_delay_model_m = actual_d_delay_m + v2_actual_mps * extra_after_actual_t2_s
        extension_method = 'constant-v2 no-brake extension after observed t2'
    d2_model_m = d1_m - d_delay_model_m
    available_to_contact_m = d2_model_m + contact_correction_m
    required_stopping_distance_m = brake_start_speed_mps**2 / (2 * equivalent_deceleration_mps2)
    margin_model_m = available_to_contact_m - required_stopping_distance_m
    collision_model_predicted = margin_model_m < 0
    if not collision_model_predicted:
        impact_speed_model_mps = 0.0
    elif available_to_contact_m <= 0:
        impact_speed_model_mps = brake_start_speed_mps
    else:
        impact_speed_model_mps = math.sqrt(max(0.0, brake_start_speed_mps**2 - 2 * equivalent_deceleration_mps2 * available_to_contact_m))
    return {
        'scenario': name, 'added_delay_ms': added_delay_ms, 'response_latency_ms': response_latency_ms,
        'brake_start_speed_model_mps': brake_start_speed_mps, 'D_delay_model_m': d_delay_model_m,
        'D2_model_m': d2_model_m, 'available_to_contact_model_m': available_to_contact_m,
        'required_stopping_distance_model_m': required_stopping_distance_m, 'margin_model_m': margin_model_m,
        'collision_model_predicted': collision_model_predicted, 'impact_speed_model_mps': impact_speed_model_mps,
        'extension_method': extension_method,
    }

scenarios = pd.DataFrame([
    evaluate_scenario('Baseline restored', 0.0),
    evaluate_scenario('+ Fusion excess only', fusion_excess_ms),
    evaluate_scenario('+ Planning excess only', planning_excess_ms),
    evaluate_scenario('+ Both, actual overlap (primary)', both_union_ms),
    evaluate_scenario('+ Both, serial upper bound', both_serial_ms),
    evaluate_scenario('Actual observed reference', actual_latency_ms - baseline_ms),
])
shown = scenarios[['scenario','added_delay_ms','response_latency_ms','D_delay_model_m','D2_model_m','margin_model_m','collision_model_predicted','impact_speed_model_mps']]
print(shown.to_string(index=False))

                     scenario  added_delay_ms  response_latency_ms  D_delay_model_m  D2_model_m  margin_model_m  collision_model_predicted  impact_speed_model_mps
            Baseline restored        0.000000           300.358057         4.867182   33.391024        5.494593                      False                0.000000
           + Fusion excess only      407.655239           708.013296        11.822433   26.435772       -6.043845                       True                7.348063
         + Planning excess only      458.496400           758.854457        12.717063   25.541143       -6.654447                       True                7.710316
+ Both, actual overlap (primary)      481.400251           781.758308        13.118896   25.139310       -6.928705                       True                7.867599
  + Both, serial upper bound      866.151639          1166.509696        19.853306   18.404899      -13.563702                       True               11.007921
    Actual obser

In [4]:
lower_ms, upper_ms = 0.0, actual_latency_ms - baseline_ms
for _ in range(80):
    midpoint_ms = (lower_ms + upper_ms) / 2
    if evaluate_scenario('threshold', midpoint_ms)['margin_model_m'] >= 0:
        lower_ms = midpoint_ms
    else:
        upper_ms = midpoint_ms
critical_added_delay_ms = (lower_ms + upper_ms) / 2
print(f'critical_added_delay_ms={critical_added_delay_ms:.6f}')
print(f'critical_total_response_ms={baseline_ms + critical_added_delay_ms:.6f}')
print(f'Fusion excess / critical = {fusion_excess_ms / critical_added_delay_ms:.3f}')
print(f'Planning excess / critical = {planning_excess_ms / critical_added_delay_ms:.3f}')

critical_added_delay_ms=195.691466
critical_total_response_ms=496.049523
Fusion excess / critical = 2.083
Planning excess / critical = 2.343


In [5]:
alternative_delays = [
    ('Fusion: gap minus gap p90', fusion_gap_ms - value('target_gap_p90_ms')),
    ('Planning: total minus p90', planning_total_ms - 35.5699),
    ('Planning: total minus 100ms', planning_total_ms - 100.0),
]
reference_sensitivity = pd.DataFrame([
    {
        'sensitivity_case': name,
        'added_delay_ms': delay,
        'margin_model_m': evaluate_scenario(name, delay)['margin_model_m'],
        'collision_model_predicted': evaluate_scenario(name, delay)['collision_model_predicted'],
    } for name, delay in alternative_delays
])
print(reference_sensitivity.to_string(index=False))

deceleration_sensitivity = []
for label, scenario_name in [
    ('Fusion only', '+ Fusion excess only'),
    ('Planning only', '+ Planning excess only'),
    ('Both, overlap', '+ Both, actual overlap (primary)'),
]:
    row = scenarios.loc[scenarios['scenario'] == scenario_name].iloc[0]
    required_decel = row['brake_start_speed_model_mps']**2 / (2 * row['available_to_contact_model_m'])
    deceleration_sensitivity.append({
        'scenario': label,
        'required_sustained_decel_mps2': required_decel,
        'increase_vs_observed_pct': (required_decel / equivalent_deceleration_mps2 - 1) * 100,
    })
deceleration_sensitivity = pd.DataFrame(deceleration_sensitivity)
print()
print(deceleration_sensitivity.to_string(index=False))

           sensitivity_case  added_delay_ms  margin_model_m  collision_model_predicted
 Fusion: gap minus gap p90      400.856304       -5.962000                       True
Planning: total minus p90      437.025100       -6.396882                       True
Planning: total minus 100ms      372.595000       -5.239436                       True

       scenario  required_sustained_decel_mps2  increase_vs_observed_pct
    Fusion only                       5.405627                 21.016260
  Planning only                       5.533656                 23.882461
Both, overlap                       5.593877                 25.230625


## Takeaways

- 原baseline恢复反事实仍然安全：模型余量 +5.495 m。
- 只加入Fusion异常超额，模型已转为碰撞，预测撞击速度7.35 m/s。
- 只加入Planning异常超额，模型已转为碰撞，预测撞击速度7.71 m/s。
- 两者按实际重叠加入是主联合结果：总响应781.758 ms，预测撞击速度7.87 m/s，接近实际7.988 m/s。
- 两者串行相加是保守上界，不代表实际事件关系。
- 关键边界约为额外195.691 ms；两项异常超额均大于该值，因此碰撞判定对正常参考的合理改变具有稳健性。
- 如果能够把整个剩余制动阶段的持续等效减速度提高约21%至25%，各插入情景可能重新避免碰撞；因此该反事实结论依赖‘保持1131实际制动能力’这一模型假设。